# Popularity Baseline 2: Most often rated positively movies

Logan Wong

law3082

According to the DRR paper, "Deep Reinforcement Learning based Recommendation with Explicit User-Item Interactions Modeling"

Popularity recommends the most popular item:
1. the item with the highest average rating 

2. the items with ratings largest number of positive from current available items to the users at each timestep.

They split each user session into 80% training / 20% testing

in offline evaluation the recommender only chooses among items in the current test session because that is the only place with ground-truth feedback.

In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import torch


In [2]:
seed = 316
random.seed(seed)
np.random.seed(seed)

In [3]:
# train_sequences_idx is a Dictionary
# 6040 keys (User indices, NOT User IDs, 0 to 6039)
# Values: list of movie indices, the first 80% of each user's ratings (all ratings 1-5, not filtered for positivity)
with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)
with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
with open('../data/idx2user.pkl', 'rb') as f:
    idx2user = pickle.load(f)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)


# ratings_dict is a Dictionary
# Each key is a tuple (user index, movie index)
# Values are that user's rating for that movie
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)
    
with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_index_to_title = pickle.load(f)

# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

So for every movie in the train set, we 
Count how many positive ratings each movie got

At test time, among the current candidate items, pick those with the highest counts, based on the counts calculated from the train set

Recommend the top 5 or 10 movies with the highest positive rating frequency.

In [4]:
# train_sequences_idx is a Dictionary
# 6040 keys (User indices, NOT User IDs, 0 to 6039)
# Values: list of movie indices, the first 80% of each user's ratings (all ratings 1-5, not filtered for positivity)

# Define positive rating threshold (e.g., 4 or 5, adjust as needed)
POSITIVE_THRESHOLD = 4

positive_counts = {}
for user_index, movie_list in train_sequences_idx.items():
    for movie_idx in movie_list:
        rating = ratings_dict[(user_index, movie_idx)]
        if rating >= POSITIVE_THRESHOLD:
            positive_counts[movie_idx] = positive_counts.get(movie_idx, 0) + 1

# Sort movies by positive count (greatest to least)
sorted_keys = sorted(positive_counts.keys(), key=lambda m: positive_counts[m], reverse=True)
popularity_ranking = {m: positive_counts[m] for m in sorted_keys}

print(f"FROM TRAIN SET, Number of movies with positive ratings: {len(popularity_ranking)}")
print(f"{'Rank':<5} | {'ID':<5} | {'Idx':<6} | {'Title':<40} | {'# of Positive Ratings':<25}")
print("-" * 90)
for i, m in enumerate(popularity_ranking):
    movie_id = idx2movie[m]
    title = movie_id_to_title[movie_id]
    pos_count = positive_counts[m]
    print(f"{i+1:<5} | {movie_id:<5} | {m:<6} | {title[:40]:<40} | {pos_count:<25}")

FROM TRAIN SET, Number of movies with positive ratings: 3465
Rank  | ID    | Idx    | Title                                    | # of Positive Ratings    
------------------------------------------------------------------------------------------
1     | 2858  | 2651   | American Beauty (1999)                   | 2635                     
2     | 260   | 253    | Star Wars: Episode IV - A New Hope (1977 | 2375                     
3     | 1196  | 1106   | Star Wars: Episode V - The Empire Strike | 2302                     
4     | 2028  | 1848   | Saving Private Ryan (1998)               | 2062                     
5     | 1198  | 1108   | Raiders of the Lost Ark (1981)           | 2056                     
6     | 593   | 579    | Silence of the Lambs, The (1991)         | 2008                     
7     | 2571  | 2374   | Matrix, The (1999)                       | 1965                     
8     | 1210  | 1120   | Star Wars: Episode VI - Return of the Je | 1951                     
9 

# Evaluation

In [5]:
# k_values = [5, 10]
# for k in k_values:

user_test_results = {}

k=10

for user_index, movie_list in test_sequences_idx.items():
    user_ID = idx2user[user_index]
    predicted_movie_ratings = {}

    for movie_idx in movie_list:
        if movie_idx in popularity_ranking:
            popularity = popularity_ranking[movie_idx]
            predicted_movie_ratings[movie_idx] = popularity
        # else:
            # print(f"{movie_idx} not in popularity_ranking")

    sorted_movies = sorted(predicted_movie_ratings.keys(), key=lambda m: predicted_movie_ratings[m], reverse=True)
    top_k_movies = sorted_movies[:k]


    top_k_movies_info = []
    for rank, movie_idx in enumerate(top_k_movies, 1):
        info = {}
        movie_id = idx2movie[movie_idx]
        movie_title = movie_index_to_title[movie_idx]
        popularity = predicted_movie_ratings[movie_idx]
        true_rating = ratings_dict.get((user_index, movie_idx), 'N/A')

        info['movie_id'] = movie_id
        info['movie_index'] = movie_idx
        info['movie_title'] = movie_title
        info['popularity'] = popularity
        info['true_rating'] = true_rating
        top_k_movies_info.append(info)
        
    user_test_results[user_ID] = top_k_movies_info

In [6]:
print(list(idx2user.keys())[-1])

print(list(user_test_results.keys())[-1])

6039
6040


In [7]:
users_to_explain = [1, 3, 40, 67, 316]   # user IDs

print(f"Top {k} movies by avg rating")
print(f"{'Rank':<5} | {'ID':<5} | {'Index':<6} | {'Title':<40} | {'Popularity':<12} | {'Actual User Rating':<12}")
print("-" * 95)
for user_id, top_k_movies  in user_test_results.items():
    user_idx = user2idx[user_id]

    if user_id in users_to_explain:
        print(f"User {user_id}:")
        for rank, info in enumerate(top_k_movies, 1):
            movie_id = info['movie_id']
            movie_idx = info['movie_index']
            movie_title = info['movie_title']
            popularity = info['popularity']
            true_rating = info['true_rating']
            print(f"{rank:<5} | {movie_id:<5} | {movie_idx:<6} | {movie_title[:40]:<40} | {popularity:<12} | {true_rating:<12}")
        print("\n")

Top 10 movies by avg rating
Rank  | ID    | Index  | Title                                    | Popularity   | Actual User Rating
-----------------------------------------------------------------------------------------------
User 1:
1     | 1     | 0      | Toy Story (1995)                         | 1455         | 5.0         
2     | 2355  | 2162   | Bug's Life, A (1998)                     | 1059         | 5.0         
3     | 588   | 574    | Aladdin (1992)                           | 776          | 4.0         
4     | 595   | 581    | Beauty and the Beast (1991)              | 603          | 5.0         
5     | 745   | 708    | Close Shave, A (1995)                    | 538          | 3.0         
6     | 2294  | 2102   | Antz (1998)                              | 267          | 4.0         
7     | 1907  | 1727   | Mulan (1998)                             | 245          | 4.0         
8     | 2687  | 2483   | Tarzan (1999)                            | 217          | 3.0        

In [8]:
def calculate_metrics(user_test_results, k_val, threshold=4):
    precisions = []
    ndcgs = []

    for user_id, movie_list in user_test_results.items():
        # Get ratings for the top K movies
        top_k = movie_list[:k_val]
        
        # Convert ratings to binary relevance (1 if >= threshold, else 0)
        # We use .get() and handle 'N/A' by defaulting to 0
        relevance = [1 if (m['true_rating'] != 'N/A' and m['true_rating'] >= threshold) else 0 for m in top_k]

        # Precision@K
        precision = sum(relevance) / k_val
        precisions.append(precision)

        # NDCG@K
        # DCG = sum of (relevance / log2(rank + 1))
        dcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevance)])
        
        # IDCG = Ideal DCG (all hits sorted at the top)
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevance)])
        
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    mean_precision = np.mean(precisions)
    mean_ndcg = np.mean(ndcgs)
    return mean_precision, mean_ndcg

In [10]:
for k in [5, 10]:
    precision_k, ndcg_k = calculate_metrics(user_test_results, k_val=k)
    # print(f"K={k}")
    print(f"Precision@{k}: {precision_k:.4f}")
    print(f"NDCG@{k}: {ndcg_k:.4f}")
    
# Precision@5: 0.6927
# Precision@10: 0.6070

# NDCG@5: 0.8617
# NDCG@10: 0.8614

Precision@5: 0.6927
NDCG@5: 0.8617
Precision@10: 0.6070
NDCG@10: 0.8614


# Average of both Baselines

In [14]:
def find_avg(a, b):
    return round((a + b)/2, 4)
    
b1_p5 = 0.7319
b1_ndcg_5 = 0.7843

b1_p10 = 0.6345
b1_ndcg_10 = 0.8008

b2_p5 = 0.6927
b2_ndcg_5 = 0.8617

b2_p10 = 0.6070
b2_ndcg_10 = 0.8614

avg_p_5 = find_avg(b1_p5, b2_p5)

avg_ndcg_5 = find_avg(b1_ndcg_5, b2_ndcg_5)

avg_p_10 = find_avg(b1_p10, b2_p10)

avg_ndcg_10 = find_avg(b1_ndcg_10, b2_ndcg_10)

print(f"Precision@5: {avg_p_5}")
print(f"Precision@10: {avg_p_10}")

print(f"NDCG@5: {avg_ndcg_5}")
print(f"NDCG@10: {avg_ndcg_10}")

Precision@5: 0.7123
Precision@10: 0.6207
NDCG@5: 0.823
NDCG@10: 0.8311
